<a href="https://colab.research.google.com/github/ibehbudova7711/CommunityDetectionAlgorithms/blob/master/ReasoningSFTv1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Install necessary libraries**

In [ ]:
# Unzip the library
!unzip unsloth.zip

# Go inside the directory
%cd /content/unsloth

# Install unsloth library with the configuration
!pip install -e ".[cu128-torch280]"

# Install flash-attention-2
!pip install flash-attn --no-build-isolation --no-cache-dir \
    --find-links https://github.com/Dao-AILab/flash-attention/releases/tag/v2.8.3

Archive:  unsloth.zip
   creating: unsloth/
  inflating: unsloth/.gitattributes  
   creating: unsloth/.github/
  inflating: unsloth/.github/FUNDING.yml  
   creating: unsloth/.github/ISSUE_TEMPLATE/
  inflating: unsloth/.github/ISSUE_TEMPLATE/bug---issue.md  
  inflating: unsloth/.github/ISSUE_TEMPLATE/feature-request.md  
   creating: unsloth/.github/workflows/
  inflating: unsloth/.github/workflows/stale.yml  
  inflating: unsloth/.github/CODEOWNERS  
  inflating: unsloth/.pre-commit-ci.yaml  
  inflating: unsloth/CODE_OF_CONDUCT.md  
  inflating: unsloth/CONTRIBUTING.md  
  inflating: unsloth/README.md       
   creating: unsloth/images/
  inflating: unsloth/images/Assistant.png  
  inflating: unsloth/images/Colab.png  
  inflating: unsloth/images/Discord button.png  
  inflating: unsloth/images/Discord.png  
  inflating: unsloth/images/Documentation Button.png  
  inflating: unsloth/images/Free version button.png  
  inflating: unsloth/images/Kaggle.png  
  inflating: unsloth/imag

# **Initialize Weights and Biases (WANDB)**

In [2]:
# Google Colab way of setting environment variables
import wandb
import os
from google.colab import userdata

# Inject api key & project to `os.environ`
#os.environ['HF_TOKEN'] = userdata.get("HF_TOKEN")
#os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
#os.environ["WANDB_PROJECT"] = userdata.get("WANDB_PROJECT")

if wandb.login():
    print("Successfully logged !")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: behbudovailahe-2001 (behbudovailahe-2001-ad-age) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Successfully logged !


In [3]:
# Default environment variables setting
import wandb
import os

if wandb.login():
    print("Successfully logged !")

Successfully logged !


# **Load Libraries**

In [4]:
import torch
from unsloth import FastLanguageModel
from trl import SFTConfig, SFTTrainer

from datasets import (load_dataset, concatenate_datasets,
                      Dataset, DatasetDict)

from tqdm.notebook import tqdm
from typing import Dict, List, Optional, Any

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# **Initialize Model**

In [5]:
# Configure model settings
model_name = "Qwen/Qwen3-4B"
MAX_SEQ_LENGTH = 128
LORA_RANK = 16

In [6]:
# Load the model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    attn_implementation="flash_attention_2",
    dtype=torch.bfloat16,
)

# Define & add special tokens to tokenizer
format_tag = [
    '<ANSWER>',   '</ANSWER>',
    '<CODE>',     '</CODE>',
    '<COT>',      '</COT>',
    '<LONG_COT>', '</LONG_COT>'
]
tokenizer.add_tokens(format_tag, special_tokens=True)

# Resize the embedding size to match tokenizer length
# model.resize_token_embeddings(len(tokenizer))

==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

unsloth/qwen3-4b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


8

In [7]:
# Apply PEFT (LORA) wrapper
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"
    ],
    lora_alpha = LORA_RANK * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:919: UserWarning: Model with `tie_word_embeddings=True` and the tied_target_modules=['lm_head'] are part of the adapter. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. See for example https://github.com/huggingface/peft/issues/2018.
  warnings.warn(
Unsloth 2026.3.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: Training embed_tokens in mixed precision to save VRAM


In [8]:
import torch
torch.cuda.empty_cache()

model.gradient_checkpointing_enable()

In [26]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True,
)

# **Dataset Loader & Tokenizer**

In [17]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B")

def tokenize(example: Dict[str, str]) -> Dict[str, List[int]]:
    # Get Instruction & Solution pairs
    instruction = example['instruction']
    solution = example['output']

    # Define prompt message
    prompt_message = [
        {'role': 'user',      'content': instruction}
    ]

    # Tokenize only prompt message
    prompt_ids = tokenizer.apply_chat_template(
        prompt_message,
        tokenize=True,
        add_generation_prompt=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH
    )['input_ids']

    # Get length of tokenized prompt message
    prompt_length = len(prompt_ids)

    # Define full conversation
    full_message = [
        {'role': 'user',      'content': instruction},
        {'role': 'assistant', 'content': solution}
    ]

    # Tokenize full conversation
    input_ids = tokenizer.apply_chat_template(
        full_message,
        tokenize=True,
        add_generation_prompt=False,
        truncation=True,
        max_length=MAX_SEQ_LENGTH
    )['input_ids']

    # Set `ignore_index` to prompt tokens to ignore during loss calculation
    labels = [-100] * prompt_length + input_ids[prompt_length:]

    # Return dictionary with corresponding fields
    return {
        'input_ids': input_ids,
        'attention_mask': [1] * len(input_ids),
        'labels': labels
    }

In [18]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B",
    max_seq_length=128,
    load_in_4bit=True
)

==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

unsloth/qwen3-4b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [19]:
# 1. Upload file
from google.colab import files
files.upload()

# 2. Imports
import pandas as pd
from datasets import Dataset

# 3. Read Excel
df = pd.read_excel("train.xlsx")

# 4. Rename columns
df = df.rename(columns={
    "original": "instruction",
    "corrected": "output"
})

# 5. Clean data
df = df.dropna()

# 6. Convert to dataset
dataset = Dataset.from_pandas(df)

# 7. Train / Val / Test split (80/10/10)
dataset = dataset.train_test_split(test_size=0.2)

train_data = dataset["train"]
temp_data  = dataset["test"]

temp_split = temp_data.train_test_split(test_size=0.5)

ds_train = train_data
ds_val   = temp_split["train"]
ds_test  = temp_split["test"]

# 8. Tokenize function (SƏNİN DATA ÜÇÜN)
def tokenize(example):
    text = f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['output']}"

    tokens = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

# 9. Apply tokenize
dst_train = ds_train.map(tokenize, remove_columns=ds_train.column_names)
dst_val   = ds_val.map(tokenize, remove_columns=ds_val.column_names)
dst_test  = ds_test.map(tokenize, remove_columns=ds_test.column_names)

Saving test.xlsx to test (4).xlsx
Saving train.xlsx to train (5).xlsx
Saving validation.xlsx to validation (3).xlsx


Map:   0%|          | 0/59408 [00:00<?, ? examples/s]

Map:   0%|          | 0/7426 [00:00<?, ? examples/s]

Map:   0%|          | 0/7427 [00:00<?, ? examples/s]

In [2]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=5e-6,
    warmup_steps=50,

    fp16=True,
    bf16=False,


    report_to="wandb",
    run_name="qwen3-test-4B",

    logging_steps=10,
    save_strategy="no",
    report_to="none"
)

ModuleNotFoundError: No module named 'trl'

# **Model training**

In [33]:
# Model training configuration
EPOCH = 3

USE_BF16 = False
BF_TAG = 'b16' if USE_BF16 else 'f16'

In [42]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=dst_train,
    eval_dataset=dst_val,
)
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 59,408 | Num Epochs = 2 | Total steps = 118,816
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 5,898,240 of 4,028,366,336 (0.15% trained)


Step,Training Loss
10,6.069319
20,5.873320
30,5.708326
40,6.226184


KeyboardInterrupt: 